# NeuroLink 4.0 — Baseline Comparison Study
### *NeuroLink V4 vs Statistical Process Control · Isolation Forest · One-Class SVM · Non-DP Autoencoder*

---

| | |
|---|---|
| **Goal** | Evaluate whether NeuroLink's anomaly detection outperforms standard alternatives |
| **Methods** | SPC · Isolation Forest · One-Class SVM · Non-DP AE · NeuroLink V4 |
| **Data** | Same 50k+ lessons, identical chronological train/test split |
| **Evaluation** | No ground-truth labels — proxy metrics used (WPM correlation, agreement, early-period detection) |

> All five methods are trained on the **same 75% training split** and evaluated on the **same full dataset**.

## 0. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','opacus>=1.4.0'], check=True)

import json, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from scipy.stats import entropy as scipy_entropy, ttest_ind
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from opacus import PrivacyEngine

warnings.filterwarnings('ignore')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print('All libraries loaded.')

## 1. Data Loading & Feature Engineering
*(Identical to NeuroLink V4 notebook)*

In [ ]:
DATA_PATH = 'transformed_typing_features_v4.csv'

ALL_CHARS       = ['space'] + list('abcdefghijklmnopqrstuvwxyz')
LEFT_KEYS       = list('qwertasdfgzxcvb')
RIGHT_KEYS      = list('yuiophjklnm')
CHAR_TIME_COLS  = [f'{c}_time'   for c in ALL_CHARS]
CHAR_ERROR_COLS = [f'{c}_errors' for c in ALL_CHARS]
CHAR_HIT_COLS   = [f'{c}_hits'   for c in ALL_CHARS]

SPEED_IDX    = [0, 1, 2, 3, 4]
ACCURACY_IDX = [5, 6, 7, 8, 9]
MOTOR_IDX    = [10, 11, 12, 13, 14]
N_SPEED = N_ACCURACY = N_MOTOR = 5

FEATURE_COLS = [
    'EV', 'RSV', 'wpm_vs_50roll', 'improvement_rate', 'warmup_effect',
    'error_entropy', 'speed_var_wl', 'worst_char_improvement', 'miss_ratio_top3', 'error_rate_trend',
    'CTA', 'CTA_cos', 'hand_asymmetry', 'time_of_day', 'speed_dispersion',
]

df_raw = pd.read_csv(DATA_PATH)
df_raw['timeStamp'] = pd.to_datetime(df_raw['timeStamp'], utc=True)
df_raw = df_raw.sort_values('timeStamp').reset_index(drop=True)
LEFT_TIME_COLS  = [f'{k}_time' for k in LEFT_KEYS  if f'{k}_time'  in df_raw.columns]
RIGHT_TIME_COLS = [f'{k}_time' for k in RIGHT_KEYS if f'{k}_time'  in df_raw.columns]
print(f'Loaded {len(df_raw):,} lessons')

def engineer_features(df_raw):
    df = df_raw.copy().sort_values('timeStamp').reset_index(drop=True)
    df['date'] = df['timeStamp'].dt.date
    df['hour'] = df['timeStamp'].dt.hour
    df['EV'] = df['wpm'] - df['wpm'].rolling(7, min_periods=1).mean()
    df['RSV'] = df['wpm'].rolling(30, min_periods=1).std().fillna(0.0)
    r50m = df['wpm'].rolling(50, min_periods=1).mean()
    r50s = df['wpm'].rolling(50, min_periods=1).std().fillna(1.0)
    df['wpm_vs_50roll'] = (df['wpm'] - r50m) / (r50s + 1e-8)
    wpm_v = df['wpm'].values
    slopes = np.zeros(len(df), dtype=np.float32)
    for i in range(len(df)):
        w = wpm_v[max(0,i-30):i+1]
        if len(w) >= 3:
            slopes[i] = float(np.polyfit(np.arange(len(w), dtype=np.float32), w, 1)[0])
    df['improvement_rate'] = slopes
    def _warmup(g):
        n = len(g)
        if n < 5: return pd.Series(0.0, index=g.index)
        n20 = max(1, n//5)
        val = (g['wpm'].iloc[-n20:].mean() - g['wpm'].iloc[:n20].mean()) / (g['wpm'].std()+1e-8)
        return pd.Series(val, index=g.index)
    df['warmup_effect'] = df.groupby('date', group_keys=False).apply(_warmup).fillna(0.0)
    avail_err = [c for c in CHAR_ERROR_COLS if c in df.columns]
    err_mat = df[avail_err].fillna(0).values.astype(float)
    df['error_entropy'] = np.array([scipy_entropy(row+1e-8) for row in err_mat])
    df['speed_var_wl'] = np.log1p(df['speed_var_wl_raw'].fillna(0))
    avail_tc = [c for c in CHAR_TIME_COLS if c in df.columns]
    df['worst_char_time'] = df[avail_tc].max(axis=1).fillna(0.0)
    df['worst_char_improvement'] = (df['worst_char_time'].rolling(30, min_periods=5).mean() - df['worst_char_time']).fillna(0.0)
    chars_ok = [c for c in ALL_CHARS if f'{c}_hits' in df.columns and f'{c}_errors' in df.columns]
    if chars_ok:
        mr = {c: df[f'{c}_errors'].fillna(0)/(df[f'{c}_hits'].fillna(0)+df[f'{c}_errors'].fillna(0)+1e-8) for c in chars_ok}
        mr_df = pd.DataFrame(mr)
        df['miss_ratio_top3'] = mr_df.apply(lambda r: r.nlargest(3).mean() if r.notna().sum()>=3 else 0.0, axis=1)
    else:
        df['miss_ratio_top3'] = 0.0
    er_v = (df['overall_errors']/df['session_length'].clip(lower=1)).values
    er_slopes = np.zeros(len(df), dtype=np.float32)
    for i in range(len(df)):
        w = er_v[max(0,i-30):i+1]
        if len(w) >= 3:
            er_slopes[i] = float(np.polyfit(np.arange(len(w), dtype=np.float32), w, 1)[0])
    df['error_rate_trend'] = er_slopes
    df['CTA']     = np.sin(2*math.pi*df['hour']/24.0)
    df['CTA_cos'] = np.cos(2*math.pi*df['hour']/24.0)
    al = [c for c in LEFT_TIME_COLS  if c in df.columns]
    ar = [c for c in RIGHT_TIME_COLS if c in df.columns]
    lm = df[al].mean(axis=1); rm = df[ar].mean(axis=1)
    df['hand_asymmetry'] = np.where(lm.isna()|rm.isna(), 1.0, lm/(rm+1e-8))
    df['time_of_day'] = df['hour']/23.0
    key_mean = df[avail_tc].mean(axis=1); key_std = df[avail_tc].std(axis=1)
    df['speed_dispersion'] = (key_std/(key_mean+1e-8)).fillna(0.0)
    keep = FEATURE_COLS + ['timeStamp','wpm','date','overall_errors']
    return df[keep].dropna().reset_index(drop=True)

df = engineer_features(df_raw)
X_all = df[FEATURE_COLS].values.astype(np.float32)
dates = pd.to_datetime(df['date'])
wpm_all = df['wpm'].values
print(f'Feature matrix : {X_all.shape}')

## 2. Identical Train / Test Split

In [ ]:
n = len(X_all)
n_tr = int(n*0.75)

X_tr_raw = X_all[:n_tr]
X_te_raw = X_all[n_tr:]

scaler = StandardScaler()
X_tr_n = scaler.fit_transform(X_tr_raw)
X_te_n = scaler.transform(X_te_raw)
X_full_n = scaler.transform(X_all)

train_end = df['timeStamp'].iloc[n_tr-1]

print(f'Train : {n_tr:,} lessons  (up to {train_end.date()})')
print(f'Test  : {n-n_tr:,} lessons')
print(f'Total : {n:,} lessons  |  Features: {X_all.shape[1]}')

# Daily WPM for proxy evaluation
daily_wpm = df.groupby('date')['wpm'].mean().reset_index()
daily_wpm.columns = ['date','avg_wpm']
daily_wpm['date'] = pd.to_datetime(daily_wpm['date'])
daily_wpm = daily_wpm.sort_values('date').reset_index(drop=True)
print(f'\nTotal days : {len(daily_wpm)}')
print(f'Train days : {(daily_wpm["date"] <= pd.Timestamp(train_end.date())).sum()}')

## 3. The Five Methods

| # | Method | Type | Privacy | Temporal-aware |
|---|---|---|---|---|
| 1 | **Statistical Process Control** | Rule-based | None | Partial (rolling) |
| 2 | **Isolation Forest** | ML — tree ensemble | None | No |
| 3 | **One-Class SVM** | ML — kernel boundary | None | No |
| 4 | **Non-DP Autoencoder** | Deep learning | ❌ None | No |
| 5 | **NeuroLink V4** | Deep learning | ✅ DP-SGD ε=10 | No |

### Evaluation metrics (no ground-truth labels)
- **Anomaly rate** — % of days flagged (ideally 5–20%)
- **WPM gap** — do flagged days have significantly lower WPM? (t-test)
- **Early-period detection** — Dec 2024 days should all be flagged (obvious: WPM was ~60, now 112)
- **Agreement matrix** — pairwise Jaccard similarity between methods
- **Reconstruction R²** — for autoencoder methods only

## Method 1 — Statistical Process Control (SPC)

In [ ]:
# Rolling 30-day mean ± 2σ on daily avg WPM
daily_wpm['roll_mean'] = daily_wpm['avg_wpm'].rolling(30, min_periods=7).mean()
daily_wpm['roll_std']  = daily_wpm['avg_wpm'].rolling(30, min_periods=7).std().fillna(1.0)
daily_wpm['spc_lo']    = daily_wpm['roll_mean'] - 2*daily_wpm['roll_std']
daily_wpm['spc_hi']    = daily_wpm['roll_mean'] + 2*daily_wpm['roll_std']
# Anomaly = below lower bound (poor performance day)
daily_wpm['spc_score'] = (daily_wpm['roll_mean'] - daily_wpm['avg_wpm']) / (daily_wpm['roll_std']+1e-8)
daily_wpm['spc_flag']  = daily_wpm['avg_wpm'] < daily_wpm['spc_lo']

n_spc = daily_wpm['spc_flag'].sum()
print(f'SPC anomaly days : {n_spc} / {len(daily_wpm)} ({100*n_spc/len(daily_wpm):.1f}%)')
print(f'First 10 flagged days:')
print(daily_wpm[daily_wpm['spc_flag']][['date','avg_wpm','roll_mean','spc_lo']].head(10).to_string(index=False))

## Method 2 — Isolation Forest

In [ ]:
np.random.seed(SEED)
iso = IsolationForest(n_estimators=200, contamination=0.10, random_state=SEED, n_jobs=-1)
iso.fit(X_tr_n)

iso_scores_full = -iso.decision_function(X_full_n)  # higher = more anomalous
iso_labels_full = iso.predict(X_full_n)              # -1 = anomaly

df['iso_score'] = iso_scores_full
df['iso_flag']  = (iso_labels_full == -1).astype(int)

# Aggregate to daily — flag day if >50% of lessons are anomalous
daily_iso = df.groupby('date').agg(
    iso_score=('iso_score','mean'),
    iso_flag =('iso_flag','mean'),
).reset_index()
daily_iso['date'] = pd.to_datetime(daily_iso['date'])
daily_iso['iso_flag_day'] = daily_iso['iso_flag'] >= 0.5

daily_wpm = daily_wpm.merge(daily_iso, on='date', how='left')

n_iso = daily_wpm['iso_flag_day'].sum()
print(f'Isolation Forest anomaly days : {n_iso} / {len(daily_wpm)} ({100*n_iso/len(daily_wpm):.1f}%)')
print(f'First 10 flagged days:')
print(daily_wpm[daily_wpm['iso_flag_day']][['date','avg_wpm']].head(10).to_string(index=False))

## Method 3 — One-Class SVM

In [ ]:
# nu ≈ expected fraction of anomalies
ocsvm = OneClassSVM(kernel='rbf', nu=0.10, gamma='scale')
ocsvm.fit(X_tr_n)

ocsvm_scores = -ocsvm.decision_function(X_full_n)
ocsvm_labels = ocsvm.predict(X_full_n)

df['ocsvm_score'] = ocsvm_scores
df['ocsvm_flag']  = (ocsvm_labels == -1).astype(int)

daily_ocsvm = df.groupby('date').agg(
    ocsvm_score=('ocsvm_score','mean'),
    ocsvm_flag =('ocsvm_flag','mean'),
).reset_index()
daily_ocsvm['date'] = pd.to_datetime(daily_ocsvm['date'])
daily_ocsvm['ocsvm_flag_day'] = daily_ocsvm['ocsvm_flag'] >= 0.5

daily_wpm = daily_wpm.merge(daily_ocsvm, on='date', how='left')

n_oc = daily_wpm['ocsvm_flag_day'].sum()
print(f'One-Class SVM anomaly days : {n_oc} / {len(daily_wpm)} ({100*n_oc/len(daily_wpm):.1f}%)')
print(f'First 10 flagged days:')
print(daily_wpm[daily_wpm['ocsvm_flag_day']][['date','avg_wpm']].head(10).to_string(index=False))

## Method 4 — Non-DP Autoencoder (same architecture, no privacy)

In [ ]:
INPUT_DIM = len(FEATURE_COLS)

class BaseAutoencoder(nn.Module):
    def __init__(self, input_dim=15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.LeakyReLU(0.01),
            nn.Linear(32, 16),        nn.LeakyReLU(0.01),
            nn.Linear(16, 10),        nn.Tanh(),
        )
        self.speed_head    = nn.Sequential(nn.Linear(10,16), nn.LeakyReLU(0.01), nn.Linear(16, N_SPEED))
        self.accuracy_head = nn.Sequential(nn.Linear(10,16), nn.LeakyReLU(0.01), nn.Linear(16, N_ACCURACY))
        self.motor_head    = nn.Sequential(nn.Linear(10,16), nn.LeakyReLU(0.01), nn.Linear(16, N_MOTOR))
    def forward(self, x):
        z = self.encoder(x)
        return self.speed_head(z), self.accuracy_head(z), self.motor_head(z)

SPD_T = torch.tensor(SPEED_IDX,    device=device)
ACC_T = torch.tensor(ACCURACY_IDX, device=device)
MOT_T = torch.tensor(MOTOR_IDX,    device=device)
SPEED_W    = torch.tensor([1.0,1.0,1.0,2.0,2.0], device=device)
ACCURACY_W = torch.tensor([2.0,1.5,1.5,2.0,1.5], device=device)
MOTOR_W    = torch.tensor([1.0,1.0,2.0,1.0,1.5], device=device)
HUBER_DELTA = 1.0

def stream_loss(pred, target, weights, n_feats):
    err = (pred-target).abs()
    h = torch.where(err<HUBER_DELTA, 0.5*err**2, HUBER_DELTA*(err-0.5*HUBER_DELTA))
    return (h*weights).mean()/n_feats

torch.manual_seed(SEED)
nodp_model = BaseAutoencoder(INPUT_DIM).to(device)
X_tr_t = torch.tensor(X_tr_n, dtype=torch.float32)
nodp_loader = DataLoader(TensorDataset(X_tr_t), batch_size=64, shuffle=True)
nodp_opt = optim.Adam(nodp_model.parameters(), lr=2e-3, weight_decay=1e-5)
nodp_sched = optim.lr_scheduler.ReduceLROnPlateau(nodp_opt, mode='min', factor=0.5, patience=5)

X_vl_t = torch.tensor(scaler.transform(X_all[int(n*0.75):int(n*0.85)]), dtype=torch.float32)
EPOCHS_NODP = 120; PATIENCE_NODP = 20
best_val = float('inf'); best_state = None; pat = 0; best_ep = 0

print('Training Non-DP Autoencoder...')
print(f'{"Epoch":>5} | {"Train":>10} | {"Val":>10} | {"★":>3}')
print('-'*35)

for epoch in range(EPOCHS_NODP):
    nodp_model.train(); bl = []
    for (xb,) in nodp_loader:
        xb = xb.to(device)
        nodp_opt.zero_grad()
        sr,ar,mr = nodp_model(xb)
        loss = (stream_loss(sr,xb[:,SPD_T],SPEED_W,N_SPEED) +
                stream_loss(ar,xb[:,ACC_T],ACCURACY_W,N_ACCURACY) +
                stream_loss(mr,xb[:,MOT_T],MOTOR_W,N_MOTOR))
        loss.backward(); nodp_opt.step(); bl.append(loss.item())
    nodp_model.eval()
    with torch.no_grad():
        Xvl = X_vl_t.to(device)
        sr,ar,mr = nodp_model(Xvl)
        vl = (stream_loss(sr,Xvl[:,SPD_T],SPEED_W,N_SPEED) +
              stream_loss(ar,Xvl[:,ACC_T],ACCURACY_W,N_ACCURACY) +
              stream_loss(mr,Xvl[:,MOT_T],MOTOR_W,N_MOTOR)).item()
    nodp_sched.step(vl)
    tl = np.mean(bl)
    if vl < best_val:
        best_val=vl; best_state={k:v.cpu().clone() for k,v in nodp_model.state_dict().items()}
        best_ep=epoch; pat=0
    else:
        pat+=1
    if epoch%10==0 or epoch==EPOCHS_NODP-1:
        print(f'{epoch:>5} | {tl:>10.5f} | {vl:>10.5f} | {"★" if pat==0 else " "}')
    if pat>=PATIENCE_NODP: print(f'\n[Early stop] epoch {epoch}'); break

nodp_model.load_state_dict(best_state)
print(f'\n[Best epoch] {best_ep}  val={best_val:.5f}')

# R² evaluation
nodp_model.eval()
X_full_t = torch.tensor(X_full_n, dtype=torch.float32).to(device)
with torch.no_grad():
    sr,ar,mr = nodp_model(X_full_t)
nodp_pred = np.zeros((len(X_full_n), INPUT_DIM))
nodp_pred[:,SPEED_IDX]    = sr.cpu().numpy()
nodp_pred[:,ACCURACY_IDX] = ar.cpu().numpy()
nodp_pred[:,MOTOR_IDX]    = mr.cpu().numpy()
nodp_r2 = r2_score(X_full_n, nodp_pred, multioutput='variance_weighted')
print(f'\nNon-DP Autoencoder  Overall R² : {nodp_r2:.4f}')

# LDI scores
nodp_spd = ((sr.cpu().numpy()-X_full_n[:,SPEED_IDX   ])**2).mean(axis=1)
nodp_acc = ((ar.cpu().numpy()-X_full_n[:,ACCURACY_IDX])**2).mean(axis=1)
nodp_mot = ((mr.cpu().numpy()-X_full_n[:,MOTOR_IDX   ])**2).mean(axis=1)
nodp_ovr = (nodp_spd+nodp_acc+nodp_mot)/3.0

df['nodp_score'] = nodp_ovr
daily_nodp = df.groupby('date').agg(nodp_score=('nodp_score','mean')).reset_index()
daily_nodp['date'] = pd.to_datetime(daily_nodp['date'])
daily_wpm = daily_wpm.merge(daily_nodp, on='date', how='left')

# Threshold from train days
train_nodp = daily_wpm[daily_wpm['date'] <= pd.Timestamp(train_end.date())]
nodp_mu = train_nodp['nodp_score'].mean()
nodp_hi = np.percentile(train_nodp['nodp_score'].values, 90)
daily_wpm['nodp_flag_day'] = daily_wpm['nodp_score'] >= nodp_hi

n_nodp = daily_wpm['nodp_flag_day'].sum()
print(f'Non-DP AE anomaly days : {n_nodp} / {len(daily_wpm)} ({100*n_nodp/len(daily_wpm):.1f}%)')

## Method 5 — NeuroLink V4 (DP-SGD, ε=10.0)

In [ ]:
class ReproducibleMask:
    def __init__(self, rate=0.10, base_seed=SEED):
        self.rate=rate; self.base_seed=base_seed
    def __call__(self, x, epoch, batch_idx):
        rng = np.random.RandomState(self.base_seed+epoch*10000+batch_idx)
        mask = rng.rand(*x.shape).astype(np.float32)
        return torch.tensor(np.where(mask<self.rate,0.0,x.cpu().numpy()), device=x.device)

torch.manual_seed(SEED)
v4_model = BaseAutoencoder(INPUT_DIM).to(device)
v4_loader = DataLoader(TensorDataset(X_tr_t), batch_size=64, shuffle=True, drop_last=True)
v4_opt    = optim.Adam(v4_model.parameters(), lr=2e-3, weight_decay=1e-5)
v4_sched  = optim.lr_scheduler.ReduceLROnPlateau(v4_opt, mode='min', factor=0.5, patience=5)
masker    = ReproducibleMask()

EPOCHS_V4 = 120; TARGET_EPS = 10.0; TARGET_DELTA = 1e-5
privacy_engine = PrivacyEngine(accountant='rdp')
base_v4 = v4_model
torch.manual_seed(SEED)
v4_model, v4_opt, v4_loader = privacy_engine.make_private_with_epsilon(
    module=v4_model, optimizer=v4_opt, data_loader=v4_loader,
    epochs=EPOCHS_V4, target_epsilon=TARGET_EPS,
    target_delta=TARGET_DELTA, max_grad_norm=1.0,
)
print(f'[DP-SGD] Noise multiplier : {v4_opt.noise_multiplier:.4f}')

best_val=float('inf'); best_state=None; pat=0; best_ep=0
print(f'\nTraining NeuroLink V4 (DP-SGD)...')
print(f'{"Epoch":>5} | {"Train":>10} | {"Val":>10} | {"ε":>8} | {"★":>3}')
print('-'*45)

for epoch in range(EPOCHS_V4):
    v4_model.train(); bl=[]
    for bi,(xb,) in enumerate(v4_loader):
        xb=xb.to(device); xb_m=masker(xb,epoch,bi)
        v4_opt.zero_grad()
        sr,ar,mr=v4_model(xb_m)
        loss=(stream_loss(sr,xb[:,SPD_T],SPEED_W,N_SPEED)+
              stream_loss(ar,xb[:,ACC_T],ACCURACY_W,N_ACCURACY)+
              stream_loss(mr,xb[:,MOT_T],MOTOR_W,N_MOTOR))
        loss.backward(); v4_opt.step(); bl.append(loss.item())
    base_v4.eval()
    with torch.no_grad():
        Xvl=X_vl_t.to(device)
        sr,ar,mr=base_v4(Xvl)
        vl=(stream_loss(sr,Xvl[:,SPD_T],SPEED_W,N_SPEED)+
            stream_loss(ar,Xvl[:,ACC_T],ACCURACY_W,N_ACCURACY)+
            stream_loss(mr,Xvl[:,MOT_T],MOTOR_W,N_MOTOR)).item()
    v4_sched.step(vl)
    tl=np.mean(bl)
    if vl<best_val:
        best_val=vl; best_state={k:v.cpu().clone() for k,v in base_v4.state_dict().items()}
        best_ep=epoch; pat=0
    else: pat+=1
    if epoch%10==0 or epoch==EPOCHS_V4-1:
        eps=privacy_engine.get_epsilon(TARGET_DELTA)
        print(f'{epoch:>5} | {tl:>10.5f} | {vl:>10.5f} | {eps:>8.4f} | {"★" if pat==0 else " "}')
    if pat>=20: print(f'\n[Early stop] epoch {epoch}'); break

base_v4.load_state_dict(best_state)
final_eps = privacy_engine.get_epsilon(TARGET_DELTA)
print(f'\n[Best epoch] {best_ep}  val={best_val:.5f}  ε={final_eps:.4f}')

# R² and LDI
base_v4.eval()
with torch.no_grad():
    sr,ar,mr = base_v4(X_full_t)
v4_pred = np.zeros((len(X_full_n), INPUT_DIM))
v4_pred[:,SPEED_IDX]=sr.cpu().numpy()
v4_pred[:,ACCURACY_IDX]=ar.cpu().numpy()
v4_pred[:,MOTOR_IDX]=mr.cpu().numpy()
v4_r2 = r2_score(X_full_n, v4_pred, multioutput='variance_weighted')
print(f'\nNeuroLink V4  Overall R² : {v4_r2:.4f}')

v4_spd = ((sr.cpu().numpy()-X_full_n[:,SPEED_IDX   ])**2).mean(axis=1)
v4_acc = ((ar.cpu().numpy()-X_full_n[:,ACCURACY_IDX])**2).mean(axis=1)
v4_mot = ((mr.cpu().numpy()-X_full_n[:,MOTOR_IDX   ])**2).mean(axis=1)
v4_ovr = (v4_spd+v4_acc+v4_mot)/3.0

df['v4_score'] = v4_ovr
daily_v4 = df.groupby('date').agg(v4_score=('v4_score','mean')).reset_index()
daily_v4['date'] = pd.to_datetime(daily_v4['date'])
daily_wpm = daily_wpm.merge(daily_v4, on='date', how='left')

train_v4 = daily_wpm[daily_wpm['date'] <= pd.Timestamp(train_end.date())]
v4_hi = np.percentile(train_v4['v4_score'].values, 90)
daily_wpm['v4_flag_day'] = daily_wpm['v4_score'] >= v4_hi

n_v4 = daily_wpm['v4_flag_day'].sum()
print(f'NeuroLink V4 anomaly days : {n_v4} / {len(daily_wpm)} ({100*n_v4/len(daily_wpm):.1f}%)')

## 4. Comparison Results

In [ ]:
methods = {
    'SPC'        : 'spc_flag',
    'Iso Forest' : 'iso_flag_day',
    'OC-SVM'     : 'ocsvm_flag_day',
    'Non-DP AE'  : 'nodp_flag_day',
    'NeuroLink V4': 'v4_flag_day',
}

# ── 1. Anomaly rates ──────────────────────────────────────────────────────────
print('='*65)
print('  ANOMALY RATES')
print('='*65)
for name, col in methods.items():
    n_flag = daily_wpm[col].sum()
    pct    = 100*n_flag/len(daily_wpm)
    print(f'  {name:<15} : {n_flag:3d} / {len(daily_wpm)} ({pct:.1f}%)')

# ── 2. WPM gap (flagged vs normal) ───────────────────────────────────────────
print('\n' + '='*65)
print('  WPM GAP  (flagged days vs normal days)  — t-test')
print('='*65)
wpm_results = {}
for name, col in methods.items():
    flagged = daily_wpm[daily_wpm[col]==True ]['avg_wpm'].values
    normal  = daily_wpm[daily_wpm[col]==False]['avg_wpm'].values
    if len(flagged) > 1 and len(normal) > 1:
        t, p = ttest_ind(flagged, normal)
        wpm_results[name] = {'flagged':flagged.mean(),'normal':normal.mean(),'gap':normal.mean()-flagged.mean(),'p':p}
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
        print(f'  {name:<15}  flagged={flagged.mean():.1f} WPM  normal={normal.mean():.1f} WPM  gap={normal.mean()-flagged.mean():+.1f}  p={p:.4f} {sig}')

# ── 3. Early-period detection (Dec 2024 — obvious anomalies) ─────────────────
print('\n' + '='*65)
print('  EARLY-PERIOD DETECTION  (Dec 2024 — WPM ~60, now ~113)')
print('='*65)
dec24 = daily_wpm[daily_wpm['date'].dt.to_period('M') == '2024-12']
print(f'  Dec 2024 days: {len(dec24)}')
for name, col in methods.items():
    caught = dec24[col].sum()
    print(f'  {name:<15} : caught {caught} / {len(dec24)} ({100*caught/max(len(dec24),1):.0f}%)')

# ── 4. Agreement matrix (Jaccard) ────────────────────────────────────────────
print('\n' + '='*65)
print('  AGREEMENT MATRIX  (Jaccard similarity)')
print('='*65)
method_names = list(methods.keys())
cols         = list(methods.values())
jacc = np.zeros((len(cols), len(cols)))
for i in range(len(cols)):
    for j in range(len(cols)):
        a = daily_wpm[cols[i]].values.astype(bool)
        b = daily_wpm[cols[j]].values.astype(bool)
        intersection = (a & b).sum()
        union        = (a | b).sum()
        jacc[i,j] = intersection/union if union > 0 else 1.0

print(f'  {"":15}', end='')
for n in method_names: print(f'{n:>13}', end='')
print()
for i, n in enumerate(method_names):
    print(f'  {n:<15}', end='')
    for j in range(len(cols)): print(f'{jacc[i,j]:>13.3f}', end='')
    print()

# ── 5. R² comparison (AE methods only) ───────────────────────────────────────
print('\n' + '='*65)
print('  RECONSTRUCTION R²  (autoencoder methods only)')
print('='*65)
print(f'  Non-DP Autoencoder : {nodp_r2:.4f}')
print(f'  NeuroLink V4 (DP)  : {v4_r2:.4f}')
print(f'  Privacy cost (ΔR²) : {nodp_r2-v4_r2:+.4f}')
print('='*65)

## 5. Comparison Dashboard

In [ ]:
C_BG='#0a0a0f'; C_PANEL='#10101a'; C_CYAN='#00e5ff'
C_YELLOW='#ffd700'; C_PURPLE='#b388ff'; C_GREEN='#69ff47'
C_ORANGE='#ff9800'; C_WHITE='#e0e0e0'; C_RED='#ff6b6b'
METHOD_COLORS = [C_YELLOW, C_ORANGE, C_PURPLE, C_GREEN, C_CYAN]

fig = plt.figure(figsize=(20, 24), facecolor=C_BG)
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: Anomaly timeline — all 5 methods
ax1 = fig.add_subplot(gs[0,:])
ax1.set_facecolor(C_PANEL)
flag_cols = ['spc_flag','iso_flag_day','ocsvm_flag_day','nodp_flag_day','v4_flag_day']
for i,(name,col,c) in enumerate(zip(method_names, flag_cols, METHOD_COLORS)):
    flagged_dates = daily_wpm[daily_wpm[col]==True]['date']
    ax1.scatter(flagged_dates, [i]*len(flagged_dates), color=c, s=20, alpha=0.7, label=name)
ax1.set_yticks(range(len(method_names)))
ax1.set_yticklabels(method_names, color=C_WHITE, fontsize=10)
ax1.set_title('Anomaly Timeline — All 5 Methods', color=C_CYAN, fontweight='bold', fontsize=13)
ax1.set_xlabel('Date', color=C_WHITE)
ax1.tick_params(colors=C_WHITE); ax1.grid(alpha=0.1, color='#333355', axis='x')
for sp in ax1.spines.values(): sp.set_color('#333355')

# Panel 2: Anomaly rates bar chart
ax2 = fig.add_subplot(gs[1,0])
ax2.set_facecolor(C_PANEL)
rates = [daily_wpm[c].sum()/len(daily_wpm)*100 for c in flag_cols]
bars  = ax2.bar(method_names, rates, color=METHOD_COLORS, edgecolor='#333355', alpha=0.9)
ax2.axhline(10, color=C_RED, ls='--', lw=1.5, label='10% reference')
ax2.axhline(20, color=C_ORANGE, ls='--', lw=1.0, label='20% reference')
for bar, rate in zip(bars, rates):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{rate:.1f}%',
             ha='center', va='bottom', color=C_WHITE, fontsize=9, fontweight='bold')
ax2.set_title('Anomaly Rate per Method', color=C_YELLOW, fontweight='bold')
ax2.set_ylabel('% Days Flagged', color=C_WHITE)
ax2.tick_params(colors=C_WHITE, labelsize=9); ax2.grid(alpha=0.12, color='#333355', axis='y')
for sp in ax2.spines.values(): sp.set_color('#333355')
ax2.legend(framealpha=0.2, facecolor='#1a1a2e', edgecolor='#333355', labelcolor=C_WHITE, fontsize=8)

# Panel 3: WPM gap
ax3 = fig.add_subplot(gs[1,1])
ax3.set_facecolor(C_PANEL)
gaps = [wpm_results[n]['gap'] if n in wpm_results else 0 for n in method_names]
ps   = [wpm_results[n]['p']   if n in wpm_results else 1 for n in method_names]
bar_colors = [C_GREEN if g>5 else (C_YELLOW if g>0 else C_RED) for g in gaps]
bars = ax3.bar(method_names, gaps, color=bar_colors, edgecolor='#333355', alpha=0.9)
for bar, gap, p in zip(bars, gaps, ps):
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    ax3.text(bar.get_x()+bar.get_width()/2, max(gap,0)+0.1, f'{gap:.1f}
{sig}',
             ha='center', va='bottom', color=C_WHITE, fontsize=8, fontweight='bold')
ax3.axhline(0, color=C_WHITE, lw=0.5, alpha=0.5)
ax3.set_title('WPM Gap: Normal − Flagged Days
(higher = method flags genuinely bad days)', color=C_YELLOW, fontweight='bold', fontsize=10)
ax3.set_ylabel('WPM Difference', color=C_WHITE)
ax3.tick_params(colors=C_WHITE, labelsize=9); ax3.grid(alpha=0.12, color='#333355', axis='y')
for sp in ax3.spines.values(): sp.set_color('#333355')

# Panel 4: Agreement heatmap
ax4 = fig.add_subplot(gs[2,0])
ax4.set_facecolor(C_PANEL)
im = ax4.imshow(jacc, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax4).ax.yaxis.label.set_color(C_WHITE)
ax4.set_xticks(range(len(method_names))); ax4.set_xticklabels(method_names, rotation=30, ha='right', color=C_WHITE, fontsize=8)
ax4.set_yticks(range(len(method_names))); ax4.set_yticklabels(method_names, color=C_WHITE, fontsize=8)
for i in range(len(method_names)):
    for j in range(len(method_names)):
        ax4.text(j, i, f'{jacc[i,j]:.2f}', ha='center', va='center',
                 color='black' if jacc[i,j]>0.5 else C_WHITE, fontsize=9, fontweight='bold')
ax4.set_title('Agreement Matrix (Jaccard Similarity)', color=C_YELLOW, fontweight='bold')

# Panel 5: Early period detection
ax5 = fig.add_subplot(gs[2,1])
ax5.set_facecolor(C_PANEL)
dec_rates = [dec24[c].sum()/max(len(dec24),1)*100 for c in flag_cols]
bars = ax5.bar(method_names, dec_rates, color=METHOD_COLORS, edgecolor='#333355', alpha=0.9)
for bar, r in zip(bars, dec_rates):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{r:.0f}%',
             ha='center', va='bottom', color=C_WHITE, fontsize=9, fontweight='bold')
ax5.axhline(100, color=C_GREEN, ls='--', lw=1.5, alpha=0.7, label='100% target')
ax5.set_title('Dec 2024 Detection Rate
(obvious anomalies — WPM was ~60)', color=C_YELLOW, fontweight='bold', fontsize=10)
ax5.set_ylabel('% Dec 2024 days caught', color=C_WHITE)
ax5.set_ylim(0, 115)
ax5.tick_params(colors=C_WHITE, labelsize=9); ax5.grid(alpha=0.12, color='#333355', axis='y')
for sp in ax5.spines.values(): sp.set_color('#333355')
ax5.legend(framealpha=0.2, facecolor='#1a1a2e', edgecolor='#333355', labelcolor=C_WHITE)

# Panel 6: R² comparison + privacy cost
ax6 = fig.add_subplot(gs[3,:])
ax6.set_facecolor(C_PANEL)
ae_names  = ['Non-DP Autoencoder', 'NeuroLink V4 (DP ε=10)']
ae_r2s    = [nodp_r2, v4_r2]
ae_colors = [C_ORANGE, C_CYAN]
bars = ax6.barh(ae_names, ae_r2s, color=ae_colors, edgecolor='#333355', height=0.4, alpha=0.9)
ax6.axvline(0.90, color=C_RED, ls='--', lw=2, label='0.90 target')
for bar, r2 in zip(bars, ae_r2s):
    ax6.text(r2+0.002, bar.get_y()+bar.get_height()/2, f'R²={r2:.4f}',
             va='center', color=C_WHITE, fontsize=11, fontweight='bold')
ax6.set_xlim(0.8, 1.02)
ax6.set_title(f'Reconstruction R²: Non-DP vs DP Autoencoder  |  Privacy cost ΔR²={nodp_r2-v4_r2:+.4f}',
              color=C_CYAN, fontweight='bold', fontsize=12)
ax6.set_xlabel('R² Score', color=C_WHITE)
ax6.tick_params(colors=C_WHITE, labelsize=11); ax6.grid(alpha=0.12, color='#333355', axis='x')
for sp in ax6.spines.values(): sp.set_color('#333355')
ax6.legend(framealpha=0.2, facecolor='#1a1a2e', edgecolor='#333355', labelcolor=C_WHITE)

fig.suptitle('NEUROLINK V4 — BASELINE COMPARISON STUDY
NeuroLink vs SPC · Isolation Forest · One-Class SVM · Non-DP Autoencoder',
             color=C_CYAN, fontsize=14, fontweight='bold')
plt.savefig('neurolink_baseline_comparison.png', dpi=120, bbox_inches='tight', facecolor=C_BG)
plt.show()
print('Saved -> neurolink_baseline_comparison.png')

## 6. Summary Report

In [ ]:
print('='*68)
print('   NEUROLINK V4 — BASELINE COMPARISON REPORT')
print('='*68)
print(f'  Dataset  : {len(df):,} lessons  |  {len(daily_wpm)} days')
print(f'  Split    : 75% train / 25% test  (chronological)')
print()
print(f'  {"Method":<18} {"Anomaly%":>9} {"WPM Gap":>9} {"Dec2024%":>9} {"p-value":>10}')
print('  ' + '-'*58)
for name, col in methods.items():
    rate = 100*daily_wpm[col].sum()/len(daily_wpm)
    gap  = wpm_results.get(name,{}).get('gap', 0)
    pval = wpm_results.get(name,{}).get('p', 1)
    dec  = dec24[col].sum()/max(len(dec24),1)*100
    sig  = '***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else 'ns'))
    print(f'  {name:<18} {rate:>8.1f}% {gap:>+9.1f} {dec:>8.0f}%  {pval:>8.4f} {sig}')
print()
print(f'  Reconstruction R²:')
print(f'    Non-DP Autoencoder : {nodp_r2:.4f}')
print(f'    NeuroLink V4 (DP)  : {v4_r2:.4f}  (ε={final_eps:.2f})')
print(f'    Privacy cost ΔR²   : {nodp_r2-v4_r2:+.4f}')
print()
print(f'  Agreement (Jaccard — NeuroLink V4 vs others):')
for i, name in enumerate(method_names[:-1]):
    print(f'    NeuroLink V4 ↔ {name:<15} : {jacc[-1,i]:.3f}')
print('='*68)